# Common Distributions — companion notebook

Runnable companion for [`common-distributions.md`](./common-distributions.md).

We plot PDFs/PMFs for each distribution in the reference table, then verify the Bernoulli→Binomial→Normal limit numerically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

## 1. Discrete distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))

k = np.arange(0, 21)
axes[0].bar(k, stats.binom.pmf(k, n=20, p=0.3), color='tab:blue')
axes[0].set_title('Binomial(n=20, p=0.3)')

axes[1].bar(k, stats.poisson.pmf(k, mu=4), color='tab:orange')
axes[1].set_title('Poisson(lambda=4)')

k2 = np.arange(1, 16)
axes[2].bar(k2, stats.geom.pmf(k2, p=0.3), color='tab:green')
axes[2].set_title('Geometric(p=0.3)')

for ax in axes:
    ax.set_xlabel('k'); ax.set_ylabel('P(X=k)')
plt.tight_layout(); plt.show()

## 2. Continuous distributions

In [ ]:
x = np.linspace(-4, 4, 400)
x_pos = np.linspace(0.01, 5, 400)
x_01 = np.linspace(0.001, 0.999, 400)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

axes[0, 0].plot(x, stats.norm.pdf(x, 0, 1), label='N(0, 1)')
axes[0, 0].plot(x, stats.norm.pdf(x, 1, 0.5), label='N(1, 0.25)')
axes[0, 0].set_title('Normal'); axes[0, 0].legend()

axes[0, 1].plot(x_pos, stats.expon.pdf(x_pos, scale=1.0), label='lambda=1')
axes[0, 1].plot(x_pos, stats.expon.pdf(x_pos, scale=0.5), label='lambda=2')
axes[0, 1].set_title('Exponential'); axes[0, 1].legend()

for alpha, beta in [(2, 2), (5, 1), (1, 3)]:
    axes[0, 2].plot(x_pos, stats.gamma.pdf(x_pos, a=alpha, scale=1/beta), label=f'Gamma({alpha},{beta})')
axes[0, 2].set_title('Gamma'); axes[0, 2].legend()

for alpha, beta in [(2, 2), (5, 2), (0.5, 0.5), (2, 5)]:
    axes[1, 0].plot(x_01, stats.beta.pdf(x_01, alpha, beta), label=f'Beta({alpha},{beta})')
axes[1, 0].set_title('Beta'); axes[1, 0].legend()

for mu, sigma in [(0, 0.5), (0, 1.0), (1, 0.5)]:
    axes[1, 1].plot(x_pos, stats.lognorm.pdf(x_pos, s=sigma, scale=np.exp(mu)), label=f'Lognormal(mu={mu}, s={sigma})')
axes[1, 1].set_title('Log-normal'); axes[1, 1].legend()

axes[1, 2].plot(x, stats.norm.pdf(x, 0, 1), label='Normal (nu=inf)', lw=2)
for nu in [1, 3, 10]:
    axes[1, 2].plot(x, stats.t.pdf(x, df=nu), label=f't, nu={nu}')
axes[1, 2].set_title('Student-t — heavier tails for smaller nu')
axes[1, 2].legend()

for ax in axes.flat:
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
plt.tight_layout(); plt.show()

## 3. The Bernoulli → Binomial → Normal limit (CLT preview)

A sum of $n$ iid Bernoulli$(p)$ is Binomial$(n, p)$, which is approximately $\mathcal{N}(np, np(1-p))$ for large $n$.

In [ ]:
p = 0.3
ns = [5, 20, 100, 500]
fig, axes = plt.subplots(1, len(ns), figsize=(14, 3))

for ax, n in zip(axes, ns):
    k = np.arange(0, n + 1)
    ax.bar(k, stats.binom.pmf(k, n=n, p=p), alpha=0.6, label='Binomial PMF')
    mu = n * p
    sigma = np.sqrt(n * p * (1 - p))
    xs = np.linspace(0, n, 400)
    ax.plot(xs, stats.norm.pdf(xs, mu, sigma), 'r-', lw=2, label='Normal approx')
    ax.set_title(f'n = {n}')
    ax.set_xlabel('k'); ax.legend()
plt.tight_layout(); plt.show()

Past $n \approx 100$ the Binomial PMF is visually indistinguishable from a Normal PDF — the CLT in action.

## 4. Conjugate update: Beta–Bernoulli

Posterior over a coin's head-probability after observing flips. Watch how the posterior tightens as data accumulates.

In [ ]:
# True coin
true_p = 0.7
flips = rng.binomial(1, true_p, size=200)

# Beta(1, 1) prior (uniform)
alpha0, beta0 = 1, 1
stops = [0, 5, 20, 100, 200]
xs = np.linspace(0, 1, 400)

fig, ax = plt.subplots(figsize=(8, 4))
for n in stops:
    k = flips[:n].sum()
    a = alpha0 + k
    b = beta0 + n - k
    ax.plot(xs, stats.beta.pdf(xs, a, b), label=f'n={n}, posterior Beta({a},{b})')
ax.axvline(true_p, color='black', ls='--', label=f'true p = {true_p}')
ax.set_xlabel('p'); ax.set_ylabel('posterior density')
ax.set_title('Sequential Bayesian updating: Beta prior + Bernoulli likelihood')
ax.legend()
plt.show()